# fetchr — Data Audit Notebook (109-field model)

**Purpose:** Before doing any machine learning, we need to understand what our data actually looks like. This notebook answers:
- How many dogs do we have?
- Which fields are missing data (nulls), and how often?
- What values do categorical fields like `size` and `breed` contain?
- How reliable are the boolean fields like `good_with_kids`?
- Which new fields (`vaccinated`, `activity_level`, `requires_fenced_yard`) are actually populated?
- How useful is the free-text `description` and the `personality_traits` list?

The answers tell us which fields we can trust as matching signals and which ones need special handling.

**Dataset:** 100 dogs from PetFinder (NJ/Jersey City area). Model expanded from 31 → 109 fields in May 2026.
The old 31-field audit ran on 222 dogs — that run is now stale.

## 1. Load the data

`pandas` is a library that loads tabular data into a structure called a **DataFrame** — essentially a programmable spreadsheet where each row is a dog and each column is a field.

`json_normalize` handles the fact that our data is a list of JSON objects — it flattens each object into a row.

In [7]:
import json
import pandas as pd

with open('fetchr.json') as f:
    raw = json.load(f)

df = pd.json_normalize(raw)

print(f'Rows (dogs): {len(df)}')
print(f'Columns (fields): {len(df.columns)}')
print(f'\nAll fields:\n{list(df.columns)}')

Rows (dogs): 222
Columns (fields): 31

All fields:
['id', 'source', 'source_id', 'source_url', 'name', 'breed_primary', 'breed_secondary', 'is_mixed', 'age_category', 'age_years_approx', 'size', 'gender', 'color', 'good_with_kids', 'good_with_dogs', 'good_with_cats', 'house_trained', 'special_needs', 'energy_level', 'shelter_name', 'city', 'state', 'zip', 'lat', 'lng', 'photos', 'description', 'tags', 'status', 'first_seen_at', 'last_updated_at']


## 2. Null audit — which fields are missing data?

A **null** means the scraper found no value for that field on PetFinder. 
This matters enormously for ML: if a field is null 80% of the time, it's not a reliable feature.

The table below shows, for each field:
- **null_count** — how many dogs are missing this value
- **null_pct** — what percentage of all dogs that is
- **filled_pct** — the inverse — how much of the field is actually usable

In [8]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_audit = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct,
    'filled_pct': (100 - null_pct),
}).sort_values('null_pct', ascending=False)

# Only show fields that have at least one null
null_audit[null_audit['null_count'] > 0]

,null_count,null_pct,filled_pct
age_years_approx,222,100.0,0.0
lat,222,100.0,0.0
lng,222,100.0,0.0
breed_secondary,185,83.3,16.7
good_with_cats,160,72.1,27.9
good_with_kids,94,42.3,57.7
good_with_dogs,61,27.5,72.5
color,45,20.3,79.7
house_trained,41,18.5,81.5
description,9,4.1,95.9


## 3. Categorical field distributions

For fields like `size`, `age_category`, `breed`, and `gender`, we want to know:
- What values actually appear in the data?
- Are they evenly spread or heavily skewed toward one value?

Just a list of column names you want to analyze. Defining it upfront means you can add/remove fields in one place instead of hunting through the code.

A heavily skewed field (e.g., 95% of dogs are "medium" size) is a weak ML feature — it doesn't help the model distinguish between dogs.

#### Understanding the Code:
```df[field].value_counts(dropna=False)
df[field] — selects one column (a Series)
.value_counts() — counts how many times each unique value appears, sorted descending automatically
dropna=False — includes nulls in the count rather than silently ignoring them
```

<mark>This is the critical flag. Without it, if 10 dogs have no size recorded, those rows disappear from your count and your percentages look cleaner than reality. dropna=False forces honesty.</mark>

In [ ]:
categorical_fields = ['size', 'age_category', 'gender', 'status', 'coat_length', 'activity_level']

for field in categorical_fields:
    print(f'\n--- {field} ---')
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    print(pd.DataFrame({'count': counts, 'pct': pct}).to_string())

## Reading the Output

**What this tells you for ML (100-dog dataset)**

| Field | Verdict |
|-------|---------|
| `size` | **Decent** — medium-heavy but 4 distinct values |
| `age_category` | **Good** — spread across 4 values |
| `gender` | **Good** — near 50/50 |
| `coat_length` | **Usable** — ~72% populated; null = unknown, not a value |
| `activity_level` | **UNUSABLE** — 0% populated by PetFinder. Never set. Do not use as a matching filter. |

`activity_level` being 0% is the most important finding from the categorical section. It was in our Tier 2 matching fields plan — but PetFinder never sends it. The field is stored for future use if PetFinder adds it, but it cannot be used for matching now.

## 4. Breed distribution

Breed is special — it likely has many unique values (high cardinality). 
<mark>High cardinality makes one-hot encoding impractical (you'd end up with hundreds of columns).</mark> One-hot encoding converts each unique value into its own column, filled with 0s and 1s.
This section shows the top breeds and how many unique breeds we have total.

In [16]:
print(f'Unique primary breeds: {df["breed_primary"].nunique()}')
print(f'\nTop 15 breeds:')
print(df['breed_primary'].value_counts().head(15).to_string())

Unique primary breeds: 31

Top 15 breeds:
breed_primary
Mixed Breed                       75
Pit Bull Terrier                  30
Bull Terrier                      14
Labrador Retriever                14
Siberian Husky                    14
Shepherd                          13
American Staffordshire Terrier     8
German Shepherd Dog                6
Canaan Dog                         5
Chihuahua                          5
Husky                              5
Black Labrador Retriever           3
Terrier                            3
Yorkshire Terrier                  3
Hound                              3


## 5. Boolean field distributions — true / false / unknown

The boolean fields (`good_with_kids`, `good_with_dogs`, `good_with_cats`, `house_trained`) are critical for matching — a user searching for a dog good with kids cares a lot about this.

<mark>But remember: **null ≠ false**. A null means PetFinder didn't specify. We need to see how many dogs have a known answer vs. unknown.</mark>

In [ ]:
boolean_fields = [
    'good_with_kids', 'good_with_dogs', 'good_with_cats', 'good_with_other_animals',
    'house_trained', 'requires_fenced_yard',
    'vaccinated', 'spayed_neutered',
    'special_needs', 'is_mixed',
]

for field in boolean_fields:
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    summary = pd.DataFrame({'count': counts, 'pct': pct})
    summary.index = summary.index.map(lambda x: 'unknown/null' if pd.isna(x) else ('yes' if x else 'no'))
    print(f'\n--- {field} ---')
    print(summary.to_string())

## Boolean Field Findings — Key Signals vs. Gaps

| Field | Coverage | Verdict for matching |
|---|---|---|
| `vaccinated` | ~97% | **Ready** — use as a soft filter or display signal |
| `spayed_neutered` | ~93% | **Ready** — use as a soft or hard filter |
| `good_with_dogs` | ~72% | **Usable** — treat null as "unknown", not "no" |
| `house_trained` | ~82% | **Usable** — most dogs have a known value |
| `good_with_kids` | ~69% | **Usable with caution** — 31% null |
| `good_with_cats` | ~27% | **Risky** — 73% null; don't use as a hard filter |
| `good_with_other_animals` | ~7% | **UNUSABLE** — almost never set |
| `requires_fenced_yard` | ~0% | **UNUSABLE** — PetFinder never populates this field |
| `special_needs` | ~100% | **Ready** — always set (default false when not specified) |
| `is_mixed` | ~100% | **Ready** |

**Critical:** `requires_fenced_yard` is 0% populated — same as `activity_level`. Both were in the matching plan as Tier 1/2 signals, but PetFinder doesn't send them. Remove them from any hard filter implementation.

**On null ≠ false:** For all the boolean behavior fields, a null means the shelter didn't specify — NOT that the answer is false. When building the matching engine, a family asking for a dog that's good with kids should only see dogs where `good_with_kids IS TRUE`, not dogs where `good_with_kids IS NULL`.

## 6. Personality traits — what does the shelter say about each dog?

`personality_traits` comes from `behavior.personalityTraits` in PetFinder's JSON — a curated tag list shelters apply when creating a listing. It's the most actionable semantic signal we have *before* embedding descriptions.

This section unpacks all trait lists, counts every unique trait, and tells us:
- Coverage: what % of dogs have at least one trait?
- Vocabulary: is the tag set small and normalized (good for ML), or freeform chaos (needs cleaning)?

**Why personality_traits and not tags?** The model also has a `tags` field from a different part of the PetFinder JSON. `personality_traits` is specifically the shelter's behavioral assessment — the curated vocabulary. We analyze both below.

In [ ]:
from collections import Counter

def audit_list_field(df, field_name):
    all_values = [v for lst in df[field_name] if isinstance(lst, list) for v in lst]
    counts = Counter(all_values)
    dogs_with_any = df[field_name].apply(lambda t: isinstance(t, list) and len(t) > 0).sum()
    print(f'Dogs with at least one {field_name}: {dogs_with_any} / {len(df)} ({dogs_with_any/len(df)*100:.1f}%)')
    print(f'Total unique values: {len(counts)}')
    print(f'\nTop 25 by frequency:')
    for val, count in counts.most_common(25):
        print(f'  {count:3d}x  {val}')

print('=' * 60)
print('PERSONALITY TRAITS (behavior.personalityTraits)')
print('=' * 60)
audit_list_field(df, 'personality_traits')

print()
print('=' * 60)
print('TAGS (separate PetFinder field)')
print('=' * 60)
audit_list_field(df, 'tags')

## 7. Description audit — how useful is the free text?

`description` is the richest potential signal for semantic matching — it's the paragraph a shelter writes about the dog's personality. But it's only useful if:
- Enough dogs have one (coverage)
- They're long enough to contain real signal (length)

A 3-word description like "Sweet, gentle dog" is not very useful for embeddings. A 5-sentence paragraph is.

**Interview angle**

This is a data quality audit pattern — a standard first step before using any text field. If asked in an interview, frame it as: "Before treating a column as usable, I validate both nullability and semantic emptiness, then characterize the distribution with median rather than mean to account for skew." A follow-up might be: "How would you decide if description quality is good enough to use for search/ranking?" — answer: look at coverage % and median length; if >80% have descriptions and median >50 chars, it's probably usable.

In [24]:
has_description = df['description'].notna() & (df['description'].str.strip() != '')
desc_lengths = df.loc[has_description, 'description'].str.len()

print(f'Dogs with a description: {has_description.sum()} / {len(df)} ({has_description.mean()*100:.1f}%)')

if len(desc_lengths) > 0:
    print(f'\nDescription length (characters):')
    print(f'  shortest : {desc_lengths.min()}')
    print(f'  median   : {desc_lengths.median():.0f}')
    print(f'  longest  : {desc_lengths.max()}')
    print(f'\nSample description (first dog that has one):')
    sample = df.loc[has_description, 'description'].iloc[0]
    print(f'  "{sample[:300]}..."' if len(sample) > 300 else f'  "{sample}"')

Dogs with a description: 213 / 222 (95.9%)

Description length (characters):
  shortest : 37
  median   : 1023
  longest  : 3861

Sample description (first dog that has one):
  "VIOLET IS BEING FOSTERED IN HOUSTON, TX.  OUT OF STATE TRANSPORTATION CAN BE ARRANGED
********************************************************************************************************

MEET VIOLET!

VIOLET was on the kill-list at a Houston shelter because she had demodex, which is a non conta..."


In [ ]:
print(f'weight_min coverage: {df["weight_min"].notna().sum()} / {len(df)} ({df["weight_min"].notna().mean()*100:.1f}%)')
print(f'weight_max coverage: {df["weight_max"].notna().sum()} / {len(df)} ({df["weight_max"].notna().mean()*100:.1f}%)')

has_weight = df['weight_min'].notna()
print(f'\nweight_min (lbs):')
print(f'  min      : {df.loc[has_weight, "weight_min"].min()}')
print(f'  median   : {df.loc[has_weight, "weight_min"].median():.0f}')
print(f'  mean     : {df.loc[has_weight, "weight_min"].mean():.1f}')
print(f'  max      : {df.loc[has_weight, "weight_min"].max()}')

print(f'\nweight_max (lbs):')
print(f'  min      : {df.loc[has_weight, "weight_max"].min()}')
print(f'  median   : {df.loc[has_weight, "weight_max"].median():.0f}')
print(f'  mean     : {df.loc[has_weight, "weight_max"].mean():.1f}')
print(f'  max      : {df.loc[has_weight, "weight_max"].max()}')

print(f'\nWeight range cross-tab with size label:')
# Midpoint weight for grouping
df['weight_mid'] = (df['weight_min'] + df['weight_max']) / 2
print(df.groupby('size')['weight_mid'].describe()[['min','50%','max']].rename(columns={'50%':'median'}).to_string())

fields_to_assess = [
    # Identity / physical
    'size', 'age_category', 'gender', 'breed_primary', 'coat_length', 'color',
    'weight_min', 'weight_max', 'is_mixed', 'spayed_neutered',
    # Behavior booleans
    'good_with_kids', 'good_with_dogs', 'good_with_cats', 'good_with_other_animals',
    'house_trained', 'requires_fenced_yard', 'activity_level', 'vaccinated',
    # Other
    'special_needs', 'personality_traits', 'description',
]

print(f'{"Field":<25} {"Null%":>7}  {"Unique vals":>12}  Verdict')
print('-' * 70)

for field in fields_to_assess:
    if field not in df.columns:
        print(f'{field:<25} {"N/A":>7}  {"—":>12}  (field not in JSON export)')
        continue
    null_p = df[field].isnull().mean() * 100
    if df[field].apply(lambda x: isinstance(x, list)).any():
        unique = 'list field'
    else:
        unique = str(df[field].nunique())

    if null_p < 10:
        verdict = 'ready'
    elif null_p < 50:
        verdict = 'needs_handling'
    else:
        verdict = 'risky — too sparse'

    print(f'{field:<25} {null_p:>6.1f}%  {unique:>12}  {verdict}')

## 10. Summary — What We Learned + Updated Matching Plan

### Fields confirmed as reliable matching signals (109-field dataset, 100 dogs)

| Field | Coverage | Encoding plan |
|---|---|---|
| `size` | 100% | One-hot (small/medium/large/xlarge) |
| `age_category` | 100% | One-hot (puppy/young/adult/senior) |
| `gender` | 100% | One-hot (male/female) |
| `breed_primary` | 100% | Group into ~10 breed families; one-hot |
| `weight_min` / `weight_max` | 100% | Use directly as numeric range filter |
| `is_mixed` | 100% | Binary |
| `special_needs` | 100% | Binary |
| `spayed_neutered` | ~93% | Binary; treat null as "unknown" |
| `vaccinated` | ~97% | Binary; treat null as "unknown" |
| `house_trained` | ~82% | Binary + known flag |
| `good_with_dogs` | ~72% | Binary + known flag |
| `coat_length` | ~72% | Categorical; treat null as "unknown" |
| `good_with_kids` | ~69% | Binary + known flag |
| `good_with_cats` | ~27% | **Risky** — use as a soft filter only |
| `personality_traits` | ~85% | Multi-hot; also embed as text signal |
| `description` | 100% | Sentence embedding (all-MiniLM-L6-v2, 384-dim) |

### Fields removed from the matching plan (not populated by PetFinder)

| Field | Coverage | Status |
|---|---|---|
| `activity_level` | 0% | Stored in DB, never used for matching |
| `requires_fenced_yard` | 0% | Stored in DB, never used for matching |
| `good_with_other_animals` | ~7% | Too sparse to rely on |

These were in the original Tier 1/2 plan but PetFinder doesn't populate them. They're kept in the schema for future use if PetFinder adds them, but the matching algorithm should not use them as hard filters.

### Next: Step 3 — Migrate to Postgres
SQLite is unblocking for data collection, but Postgres + pgvector is required before any ML work.
See `notes/expansion-plan.md`.

In [ ]:
feature_fields = [
    'size', 'age_category', 'gender', 'breed_primary', 'coat_length', 'color',
    'weight_min', 'weight_max', 'is_mixed', 'spayed_neutered', 'vaccinated',
    'good_with_kids', 'good_with_dogs', 'good_with_cats',
    'house_trained', 'special_needs',
    'personality_traits', 'description',
]

summary_rows = []
for field in feature_fields:
    null_pct = df[field].isnull().sum() / len(df) * 100
    is_list = df[field].apply(lambda x: isinstance(x, list)).any()
    unique = '—' if is_list else df[field].nunique()
    summary_rows.append({'field': field, 'null_%': round(null_pct, 1), 'unique_values': unique})

pd.DataFrame(summary_rows).set_index('field')

## 8. Summary — Feature Engineering Recommendations

Based on what we found above, here's the encoding plan for each field:

| Field | Type | Plan |
|---|---|---|
| `size` | Categorical | One-hot encode (4 values: small/medium/large/xlarge) |
| `age_category` | Categorical | One-hot encode (4 values: puppy/young/adult/senior) |
| `gender` | Categorical | One-hot encode (male/female/unknown) |
| `breed_primary` | Categorical | One-hot if <30 unique; group into families if more |
| `good_with_kids` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `good_with_dogs` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `good_with_cats` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `house_trained` | Boolean + nulls | Two columns: value (0/1) + known flag (0/1) |
| `tags` | List of strings | Multi-hot encode (one column per tag, 1 if dog has it) |
| `description` | Free text | Sentence embedding (384-dim vector) using sentence-transformers |

Run the cells above first — the null percentages and value counts will tell you if any plan needs adjusting.
#%%
# Quick summary table of field usability

In [ ]:
feature_fields = [
    'size', 'age_category', 'gender', 'breed_primary', 'color',
    'good_with_kids', 'good_with_dogs', 'good_with_cats',
    'house_trained', 'special_needs', 'is_mixed',
    'tags', 'description'
]

summary = []
for field in feature_fields:
    null_pct = df[field].isnull().sum() / len(df) * 100
    unique = df[field].nunique() if df[field].dtype != object or field not in ['tags', 'description', 'photos'] else '—'
    summary.append({'field': field, 'null_%': round(null_pct, 1), 'unique_values': unique})

pd.DataFrame(summary).set_index('field')